# Simplified path — Colab GPU run

Runs `src/simplified.py` end-to-end on a Colab GPU using a larger model than the local Qwen 1.5B default. Same code as local; only the model size changes.

**Before running:** Runtime → Change runtime type → GPU (T4 is enough for 7B in fp16).

Outputs are written to `colab_outputs.md` in the working directory; download from the Files panel after the run.

## 1. Clone repo & install dependencies

In [ ]:
# Cloning from b-coman fork colab-refactor branch for Colab testing.
# Once the PR is merged into dariacoman/main, switch this clone URL.
!git clone --branch phase-1-simplified-path https://github.com/dariacoman/compliance-gap-analysis.git
%cd compliance-gap-analysis

In [ ]:
# Install only what Colab does not preinstall.
# Colab already ships torch, numpy, etc. — installing requirements.txt
# would downgrade Colab torch and break the CUDA wheel alignment.
# numpy<2.2 pin: keeps ABI consistent with Colab's preinstalled scipy/sklearn.
!pip install -q sentence-transformers transformers accelerate diskcache "numpy<2.2"

## 2. Pick the model

`MODEL_ID` is read by `src/simplified.py` at import time. Uncomment exactly one line in the next cell.

| Model | Size | Fits T4? | HF gated? | Purpose |
|---|---|---|---|---|
| `Qwen/Qwen2.5-1.5B-Instruct` | 1.5B | Yes (fp16 ~3 GB) | No | Control — same as local default; exhibits FRIA leak |
| `Qwen/Qwen2.5-3B-Instruct` | 3B | Yes (fp16 ~6 GB) | No | Intermediate scale — fast on T4 |
| `Qwen/Qwen2.5-7B-Instruct` | 7B | No (CPU offload, slow) | No | Full 7B baseline; ~3 min/query on T4 |
| `google/gemma-2-2b-it` | 2.6B | Yes (fp16 ~5 GB) | **Yes** | Older Gemma family — smaller comparison point |
| `google/gemma-3-4b-it` | 4B | Yes (fp16 ~8 GB) | **Yes** | **Newer Gemma family** — direct family-vs-family comparison to Qwen 3B; tests if FRIA-leak threshold is parameter-count- or family-specific |

**Gemma access:** requires a HuggingFace account, accepting the model license at https://huggingface.co/google/gemma-3-4b-it (or the 2-2b page), and adding `HF_TOKEN` as a Colab secret (left sidebar → key icon → add `HF_TOKEN`). Skip if running Qwen only.

**Gemma 3 caveat:** Gemma 3 was released March 2025 and requires `transformers >= 4.49`. Colab's preinstalled transformers should be recent enough, but if the model load raises *"Unknown architecture: Gemma3ForCausalLM"*, run `!pip install --upgrade transformers` in a new cell, restart runtime, and re-run. (Upgrading transformers may cascade into other version conflicts; fallback is Gemma 2-2B.)

**AWQ-quantised 7B option dropped:** earlier iterations included `Qwen/Qwen2.5-7B-Instruct-AWQ` for fast 7B on T4. The required runtime libraries (`autoawq`, `gptqmodel`) had cross-incompatibilities with Colab's preinstalled stack and we removed the option. Qwen 3B is the fast-on-T4 substitute; full 7B remains available with CPU offload for the full-precision comparison.

In [ ]:
import os

# Pick exactly one — uncomment the line you want:

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'              # default — fits T4 fully, fast
# MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'          # control — same as local default
# MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'             # full 7B — needs CPU offload on T4 (~3 min/query)
# MODEL_ID = 'google/gemma-2-2b-it'                 # alt family (older), gated, needs HF_TOKEN
# MODEL_ID = 'google/gemma-3-4b-it'                 # alt family (newer, stronger), gated, needs HF_TOKEN — RECOMMENDED Gemma

os.environ['MODEL_ID'] = MODEL_ID
os.environ['PYTHONPATH'] = '/content/compliance-gap-analysis'
print(f"Selected model: {MODEL_ID}")

# Authenticate with HuggingFace if a gated model is selected.
if 'gemma' in MODEL_ID.lower() or 'llama' in MODEL_ID.lower():
    from google.colab import userdata
    try:
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
        print('HF_TOKEN loaded from Colab secrets.')
    except Exception:
        print('WARNING: gated model selected but HF_TOKEN not found in Colab secrets.')
        print('Add it via the key icon in the left sidebar, or pick a non-gated model.')

## 3. Verify GPU & import the simplified path

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('Memory:', torch.cuda.get_device_properties(0).total_memory / 1e9, 'GB')

In [ ]:
from src.simplified import analyse, LLM_MODEL_ID
print('Model in use:', LLM_MODEL_ID)

## 4. (Optional) Tweak config for this run

`simplified.CONFIG` is a Python dict loaded from `config.toml` at the repo root. Mutate it here to override defaults for this Colab session — changes take effect on the next `analyse()` call without restarting the runtime.

Three override mechanisms with clear precedence:
1. **Environment variables** (highest) — for environment-specific overrides (`MODEL_ID`, `RANKING_STRATEGY`)
2. **CONFIG dict mutation** (this cell) — for cell-driven experimentation
3. **`config.toml` file** — for committed defaults

Skip this cell to use the project defaults from `config.toml`.

In [ ]:
import src.simplified as simplified

# Examples — uncomment exactly what you want to override.
# Default behaviour (all commented out): RRF retrieval + reranker confidence display + evidence footer on.

# Disable the reranker entirely (single-stage BGE retrieval)
# simplified.CONFIG['ranking']['strategy'] = 'bge_only'

# Use cross-encoder reranking without RRF blending (rerank fully replaces BGE order)
# simplified.CONFIG['ranking']['strategy'] = 'rerank_only'

# Retrieve more chunks (passes more context to the LLM)
# simplified.CONFIG['retrieval']['top_k_reg'] = 8
# simplified.CONFIG['retrieval']['top_k_dep'] = 8

# Stricter / looser confidence threshold for the 'strong' grounding label
# simplified.CONFIG['reranker']['confidence_thresholds']['strong'] = 0.6

# Hide the retrieval evidence footer (clean demo output)
# simplified.CONFIG['output']['show_evidence'] = False

# Confirm what is currently set for this run:
print(f"Strategy:          {simplified.CONFIG['ranking']['strategy']}")
print(f"Top-k REG / DEP:   {simplified.CONFIG['retrieval']['top_k_reg']} / {simplified.CONFIG['retrieval']['top_k_dep']}")
print(f"Top-k initial:     {simplified.CONFIG['retrieval']['top_k_initial']}")
print(f"Confidence (strong / moderate): {simplified.CONFIG['reranker']['confidence_thresholds']['strong']} / {simplified.CONFIG['reranker']['confidence_thresholds']['moderate']}")
print(f"Show evidence:     {simplified.CONFIG['output']['show_evidence']}")

## 5. Run the 5 standard test queries

Verbatim text from `docs/test-queries.md`. These are the same queries run in the local baseline (`docs/test-passes/v3-qwen-1.5b-local-baseline.md`) — outputs from this notebook are directly comparable.

In [ ]:
QUERIES = {
    'Q1 multi-facet': (
        "TalentLens compliance under EU AI Act Annex III \u00a74 \u2014 am I covered on "
        "Article 13 deployer instructions, Article 14 human oversight, Article 26 logs "
        "and worker information, and the related Article 22 GDPR automated-decisions "
        "duties? Where are the gaps?"
    ),
    'Q2 red-teaming': (
        "Does our policy address the red-teaming requirements before deploying a high-"
        "risk AI system to production?"
    ),
    'Q3 Article 22 sub-clauses': (
        "How do we meet GDPR Article 22 requirements on solely automated decisions "
        "affecting candidates \u2014 explicit consent, right to obtain human intervention, "
        "right to contest the decision, and right to express their point of view?"
    ),
    'Q4 transparency': (
        "Are we doing enough on transparency for candidates assessed by TalentLens?"
    ),
    'Q5 FRIA': (
        "Have we performed a Fundamental Rights Impact Assessment under EU AI Act "
        "Article 27 for TalentLens as a deployer of an Annex III high-risk system?"
    ),
}

In [ ]:
import time

results = {}
for label, query in QUERIES.items():
    print(f'\n{"="*70}\n{label}\n{"="*70}')
    print(f'Query: {query}\n')
    t0 = time.time()
    output = analyse(query)
    elapsed = time.time() - t0
    results[label] = {'query': query, 'output': output, 'elapsed_s': elapsed}
    print(output)
    print(f'\n[{elapsed:.1f}s]')

## 6. Save outputs to a markdown file

Download `colab_outputs.md` from the Files panel afterwards. Drop it into `docs/test-passes/` in the repo with a descriptive filename, e.g. `v4-qwen-7b-colab.md`.

In [ ]:
from datetime import datetime, timezone

lines = [
    f'# Test pass \u2014 simplified path on Colab GPU',
    '',
    f'> Run date: {datetime.now(timezone.utc).strftime("%Y-%m-%d")}',
    f'> Model: `{LLM_MODEL_ID}`',
    f'> Hardware: Colab GPU ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})',
    f'> Prompt: V4 (current default in `src/simplified.py`)',
    f'> Embeddings: BAAI/bge-large-en-v1.5',
    '',
    '---',
    '',
]
for label, r in results.items():
    lines += [
        f'## {label}',
        '',
        f'**Query:** {r["query"]}',
        '',
        f'**Generation latency:** {r["elapsed_s"]:.1f}s',
        '',
        '**Output:**',
        '',
        '```',
        r['output'].rstrip(),
        '```',
        '',
        '---',
        '',
    ]

with open('colab_outputs.md', 'w') as f:
    f.write('\n'.join(lines))

print('Wrote colab_outputs.md')